In [15]:
!pip install rioxarray geopandas tqdm
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis


In [16]:
# ==========================================================
# Monthly Linear Scaling Bias Correction
# For Multiple CMIP6 Models and Multiple Scenarios
#
# Author: Hanik Lakhe (Modified)
# ==========================================================

import os
import gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ==========================================================
# USER SETTINGS
# ==========================================================

INPUT_ROOT = "Stationwise/Precipitation/Raw_GCM_Precipitation"
OUTPUT_ROOT = "Stationwise/Precipitation/BiasCorrected_LS_Precipitation"

OBS_FILE = "Stationwise/Karnali_1986_2015.xlsx"
OBS_SHEET = "Precipitation"

SCENARIOS = [
    "historical",
    "ssp245",
    "ssp585"
]

CAL_START = "1986-01-01"
CAL_END = "2014-12-31"

# ==========================================================
# READ OBSERVED DATA
# ==========================================================

print("="*70)
print("Reading Observed Dataset...")
print("="*70)

df_obs = pd.read_excel(
    OBS_FILE,
    sheet_name=OBS_SHEET
)

# Remove unnecessary columns if present
for col in ["year", "month", "days"]:
    if col in df_obs.columns:
        df_obs.drop(columns=col, inplace=True)

# Date
df_obs["Date"] = pd.to_datetime(df_obs["Date"])
df_obs.set_index("Date", inplace=True)
df_obs.index = df_obs.index.normalize()

# Ensure string columns
df_obs.columns = df_obs.columns.astype(str)

print(f"Observed stations : {len(df_obs.columns)}")
print(f"Observed days     : {len(df_obs)}")

df_obs.head(5)

Reading Observed Dataset...
Observed stations : 10
Observed days     : 10957


,201,202,203,204,205,206,210,211,214,217
Date,,,,,,,,,,
1986-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1986-01-02,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0
1986-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1986-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1986-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# ==========================================================
# FIND AVAILABLE MODELS
# ==========================================================

models = sorted([
    d for d in os.listdir(INPUT_ROOT)
    if os.path.isdir(os.path.join(INPUT_ROOT, d))
])

print("\nModels Found")

for m in models:
    print("  ", m)

print("\nTotal Models :", len(models))


Models Found
   ACCESS-CM2

Total Models : 1


In [18]:
summary = []
# ==========================================================
# LOOP OVER MODELS
# ==========================================================
for model in tqdm(models,
                  desc="Models",
                  colour="green"):
    print(f"MODEL : {model}")
    hist_file = os.path.join(
        INPUT_ROOT,
        model,
        "historical",
        f"{model}_historical_karnali_pr.csv"
    )
    if not os.path.exists(hist_file):

        print("Historical file missing.")
        continue

    # ------------------------------------------------------
    # Read Historical GCM
    # ------------------------------------------------------

    df_hist = pd.read_csv(hist_file)

    df_hist["Date"] = pd.to_datetime(df_hist["Date"])
    df_hist.set_index("Date", inplace=True)
    df_hist.index = df_hist.index.normalize()

    df_hist.columns = df_hist.columns.astype(str)

    # ------------------------------------------------------
    # Common Dates
    # ------------------------------------------------------

    common_dates = df_obs.index.intersection(df_hist.index)

    obs = df_obs.loc[common_dates].copy()
    hist = df_hist.loc[common_dates].copy()

    print("Common Dates :", len(common_dates))

    # ------------------------------------------------------
    # Calibration Period
    # ------------------------------------------------------

    obs_cal = obs.loc[CAL_START:CAL_END].copy()
    hist_cal = hist.loc[CAL_START:CAL_END].copy()

    print("Calibration Days :", len(obs_cal))

    # ------------------------------------------------------
    # Automatic Station Matching
    # ------------------------------------------------------

    station_map = {}

    for obs_station in obs.columns:

        for gcm_station in hist.columns:

            if gcm_station.startswith(obs_station):

                station_map[obs_station] = gcm_station
                break

    print("Matched Stations :", len(station_map))

    # ------------------------------------------------------
    # Add Year and Month
    # ------------------------------------------------------

    obs_cal["year"] = obs_cal.index.year
    obs_cal["month"] = obs_cal.index.month

    hist_cal["year"] = hist_cal.index.year
    hist_cal["month"] = hist_cal.index.month

    # ------------------------------------------------------
    # Compute Monthly Correction Factors
    # ------------------------------------------------------

    correction_factors = {}

    for st in tqdm(
            station_map.keys(),
            desc="Computing CF",
            leave=False,
            colour="yellow"):

        gcm_col = station_map[st]

        obs_monthly = (
            obs_cal
            .groupby(["year", "month"])[st]
            .sum()
            .reset_index()
        )

        gcm_monthly = (
            hist_cal
            .groupby(["year", "month"])[gcm_col]
            .sum()
            .reset_index()
        )

        obs_clim = (
            obs_monthly
            .groupby("month")[st]
            .mean()
        )

        gcm_clim = (
            gcm_monthly
            .groupby("month")[gcm_col]
            .mean()
        )

        cf = obs_clim / gcm_clim.replace(0, np.nan)

        correction_factors[st] = cf

    print("Monthly correction factors computed.")
        # ======================================================
    # APPLY CORRECTION TO ALL SCENARIOS
    # ======================================================

    for scenario in tqdm(
            SCENARIOS,
            desc=f"{model}",
            leave=False,
            colour="blue"):

        input_file = os.path.join(
            INPUT_ROOT,
            model,
            scenario,
            f"{model}_{scenario}_karnali_pr.csv"
        )

        if not os.path.exists(input_file):

            print(f"\n{scenario} file not found.")
            continue

        print(f"\nProcessing {scenario}")

        try:

            # -------------------------------------------------
            # Read Scenario
            # -------------------------------------------------

            df = pd.read_csv(input_file)

            df["Date"] = pd.to_datetime(df["Date"])

            df.set_index("Date", inplace=True)

            df.index = df.index.normalize()

            df.columns = df.columns.astype(str)

            # -------------------------------------------------
            # Add Month Column
            # -------------------------------------------------

            df["month"] = df.index.month

            # -------------------------------------------------
            # Create Output Copy
            # -------------------------------------------------

            corrected = df.copy()

            # -------------------------------------------------
            # Apply Monthly Linear Scaling
            # -------------------------------------------------

            for st in tqdm(
                    station_map.keys(),
                    desc=f"{scenario}",
                    leave=False,
                    colour="cyan"):

                gcm_col = station_map[st]

                if gcm_col not in corrected.columns:
                    continue

                cf = correction_factors[st]

                for month in range(1, 13):

                    if month not in cf.index:
                        continue

                    mask = corrected["month"] == month

                    corrected.loc[
                        mask,
                        gcm_col
                    ] = (
                        corrected.loc[
                            mask,
                            gcm_col
                        ] * cf[month]
                    )

            # -------------------------------------------------
            # Remove Helper Column
            # -------------------------------------------------

            corrected.drop(columns="month", inplace=True)

            # -------------------------------------------------
            # Create Output Folder
            # -------------------------------------------------

            out_dir = os.path.join(
                OUTPUT_ROOT,
                model,
                scenario
            )

            os.makedirs(out_dir, exist_ok=True)

            output_file = os.path.join(
                out_dir,
                f"{model}_{scenario}_bias_corrected.csv"
            )

            corrected.to_csv(output_file)

            print(f"Saved : {output_file}")

            summary.append({
                "Model": model,
                "Scenario": scenario,
                "Stations": len(station_map),
                "Days": len(corrected),
                "Output": output_file,
                "Status": "Success"
            })

        except Exception as e:

            print(f"Error while processing {scenario}")
            print(e)

            summary.append({
                "Model": model,
                "Scenario": scenario,
                "Stations": 0,
                "Days": 0,
                "Output": "",
                "Status": "Failed"
            })

        finally:

            # Free memory
            del df

            if "corrected" in locals():
                del corrected

            gc.collect()

Models:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL : ACCESS-CM2
Common Dates : 10592
Calibration Days : 10592
Matched Stations : 10


Computing CF:   0%|          | 0/10 [00:00<?, ?it/s]

Monthly correction factors computed.


ACCESS-CM2:   0%|          | 0/3 [00:00<?, ?it/s]


Processing historical


historical:   0%|          | 0/10 [00:00<?, ?it/s]

Saved : Stationwise/Precipitation/BiasCorrected_LS_Precipitation/ACCESS-CM2/historical/ACCESS-CM2_historical_bias_corrected.csv

ssp245 file not found.

ssp585 file not found.
